# Submission Delay Integration & Aggregation

Calculate student submission delays relative to assessment deadlines and aggregate metrics for integration into master dataframe.


## 1. Load Source Data

Load studentAssessment.csv, assessments.csv, and courses.csv from ../../data/raw/


In [6]:
import pandas as pd
from pathlib import Path

# Load source data
RAW_DATA_DIR = Path('../../data/raw/')

studentAssessment = pd.read_csv(RAW_DATA_DIR / 'studentAssessment.csv')
assessments = pd.read_csv(RAW_DATA_DIR / 'assessments.csv')
courses = pd.read_csv(RAW_DATA_DIR / 'courses.csv')

print("=" * 80)
print("DATA LOADING")
print("=" * 80)
print(f"\nstudentAssessment shape: {studentAssessment.shape}")
print(f"assessments shape: {assessments.shape}")
print(f"courses shape: {courses.shape}")

print(f"\nstudentAssessment columns: {studentAssessment.columns.tolist()}")
print(f"assessments columns: {assessments.columns.tolist()}")
print(f"courses columns: {courses.columns.tolist()}")

print(f"\nMissing dates in assessments: {assessments['date'].isna().sum()}")
print(f"Total assessments: {len(assessments)}")

DATA LOADING

studentAssessment shape: (173912, 5)
assessments shape: (206, 6)
courses shape: (22, 3)

studentAssessment columns: ['id_assessment', 'id_student', 'date_submitted', 'is_banked', 'score']
assessments columns: ['code_module', 'code_presentation', 'id_assessment', 'assessment_type', 'date', 'weight']
courses columns: ['code_module', 'code_presentation', 'module_presentation_length']

Missing dates in assessments: 11
Total assessments: 206


## 2. Handle Missing Deadlines in Assessments

Identify missing date values in assessments.csv (exams). Merge with courses to get module presentation length and fill NaN dates.


In [7]:
# Merge assessments with courses to get module presentation length
assessments_merged = assessments.merge(
    courses[['code_module', 'code_presentation', 'module_presentation_length']],
    on=['code_module', 'code_presentation'],
    how='left'
)

print("=" * 80)
print("HANDLING MISSING EXAM DEADLINES")
print("=" * 80)

# Identify rows with missing dates
missing_dates = assessments_merged[assessments_merged['date'].isna()].copy()
print(f"\nRows with missing dates (exams): {len(missing_dates)}")
print(f"Sample of missing dates:")
print(missing_dates[['id_assessment', 'code_module', 'code_presentation', 'date', 'module_presentation_length', 'assessment_type']].head(11))

# Fill missing dates with end_of_module_presentation
# Use module_presentation_length as proxy for exam deadline
assessments_merged['date'] = assessments_merged['date'].fillna(assessments_merged['module_presentation_length'])

print(f"\nAfter filling: Missing dates remaining: {assessments_merged['date'].isna().sum()}")
print(f"Date range: {assessments_merged['date'].min()} to {assessments_merged['date'].max()}")

HANDLING MISSING EXAM DEADLINES

Rows with missing dates (exams): 11
Sample of missing dates:
     id_assessment code_module code_presentation  date  \
5             1757         AAA             2013J   NaN   
11            1763         AAA             2014J   NaN   
23           14990         BBB             2013B   NaN   
35           15002         BBB             2013J   NaN   
47           15014         BBB             2014B   NaN   
53           15025         BBB             2014J   NaN   
62           24290         CCC             2014B   NaN   
63           40087         CCC             2014B   NaN   
72           24299         CCC             2014J   NaN   
73           40088         CCC             2014J   NaN   
108          25368         DDD             2014J   NaN   

     module_presentation_length assessment_type  
5                           268            Exam  
11                          269            Exam  
23                          240            Exam  
35       

## 3. Calculate Submission Delays

Merge studentAssessment with assessments and calculate submission_delay = date_submitted - date. Identify late submissions where delay > 0.


In [8]:
# Merge studentAssessment with assessments on id_assessment
student_assess_with_dates = studentAssessment.merge(
    assessments_merged[['id_assessment', 'code_module', 'code_presentation', 'date']],
    on='id_assessment',
    how='left'
)

print("=" * 80)
print("CALCULATING SUBMISSION DELAYS")
print("=" * 80)

# Calculate submission delay in days
student_assess_with_dates['submission_delay'] = student_assess_with_dates['date_submitted'] - student_assess_with_dates['date']

# Identify late submissions
student_assess_with_dates['is_late'] = student_assess_with_dates['submission_delay'] > 0

print(f"\nTotal submissions: {len(student_assess_with_dates)}")
print(f"Late submissions: {student_assess_with_dates['is_late'].sum()}")
print(f"Late submission rate: {(student_assess_with_dates['is_late'].sum() / len(student_assess_with_dates) * 100):.2f}%")

print(f"\nSubmission delay statistics:")
print(student_assess_with_dates['submission_delay'].describe())

print(f"\nSample of late submissions:")
late_sample = student_assess_with_dates[student_assess_with_dates['is_late']].head()
print(late_sample[['id_student', 'id_assessment', 'date_submitted', 'date', 'submission_delay']].to_string())

CALCULATING SUBMISSION DELAYS

Total submissions: 173912
Late submissions: 49323
Late submission rate: 28.36%

Submission delay statistics:
count    173912.000000
mean        -16.694064
std          45.574098
min        -246.000000
25%          -7.000000
50%          -1.000000
75%           2.000000
max         372.000000
Name: submission_delay, dtype: float64

Sample of late submissions:
    id_student  id_assessment  date_submitted  date  submission_delay
1        28400           1752              22  19.0               3.0
3        32885           1752              26  19.0               7.0
5        45462           1752              20  19.0               1.0
17       74372           1752              22  19.0               3.0
20       91265           1752              21  19.0               2.0


## 4. Aggregate Submission Metrics

Group by id_student, code_module, code_presentation and calculate aggregated metrics: total_submissions, late_submissions_count, avg_submission_delay.


In [9]:
# Aggregate submission metrics by student-module-presentation
submission_metrics = student_assess_with_dates.groupby(
    ['id_student', 'code_module', 'code_presentation']
).agg({
    'id_assessment': 'count',  # total_submissions
    'is_late': 'sum',           # late_submissions_count
    'submission_delay': 'mean'  # avg_submission_delay
}).reset_index()

# Rename columns for clarity
submission_metrics.columns = [
    'id_student', 'code_module', 'code_presentation',
    'total_submissions', 'late_submissions_count', 'avg_submission_delay'
]

# Ensure all numeric columns are float type
submission_metrics['avg_submission_delay'] = submission_metrics['avg_submission_delay'].astype(float)

print("=" * 80)
print("AGGREGATED SUBMISSION METRICS")
print("=" * 80)

print(f"\nShape: {submission_metrics.shape}")
print(f"Columns: {submission_metrics.columns.tolist()}")
print(f"\nBasic statistics:")
print(submission_metrics[['total_submissions', 'late_submissions_count', 'avg_submission_delay']].describe())

print(f"\nSample of aggregated metrics:")
print(submission_metrics.head(10).to_string())

print(f"\nMissing values:")
print(submission_metrics.isnull().sum())

print(f"\nDescribe:")
print(submission_metrics.describe())

AGGREGATED SUBMISSION METRICS

Shape: (25843, 6)
Columns: ['id_student', 'code_module', 'code_presentation', 'total_submissions', 'late_submissions_count', 'avg_submission_delay']

Basic statistics:
       total_submissions  late_submissions_count  avg_submission_delay
count       25843.000000            25843.000000          25843.000000
mean            6.729559                1.908563            -12.224324
std             3.771381                2.308085             26.212686
min             1.000000                0.000000           -236.000000
25%             4.000000                0.000000            -11.000000
50%             7.000000                1.000000             -1.000000
75%            10.000000                4.000000              1.090909
max            14.000000               12.000000            187.000000

Sample of aggregated metrics:
   id_student code_module code_presentation  total_submissions  late_submissions_count  avg_submission_delay
0        6516         

## 5. Merge and Export Integrated Data

Integrate aggregated submission metrics into the existing master dataframe. Fill NaN values with 0 and save to data/merged/merged.csv.


In [10]:
# Load the existing merged dataframe
MERGED_DIR = Path('../../data/merged')
MERGED_FILE = MERGED_DIR / 'merged.csv'

print("=" * 80)
print("MERGING SUBMISSION METRICS INTO MASTER DATAFRAME")
print("=" * 80)

# Check if merged file exists
if MERGED_FILE.exists():
    master = pd.read_csv(MERGED_FILE)
    print(f"\nLoaded existing merged dataframe: {master.shape}")
else:
    print(f"\nMerged file not found at {MERGED_FILE}")
    print("Loading from data_integration.ipynb output...")
    # This should be run after data_integration.ipynb
    raise FileNotFoundError(f"Master dataframe not found at {MERGED_FILE}")

print(f"Master columns before merge: {master.columns.tolist()}")

# Merge submission metrics into master
master_updated = master.merge(
    submission_metrics,
    on=['id_student', 'code_module', 'code_presentation'],
    how='left'
)

print(f"\nShape after merge: {master_updated.shape}")
print(f"New columns: {[col for col in master_updated.columns if col not in master.columns]}")

# Fill NaN values with 0 for submission metrics
submission_cols = ['total_submissions', 'late_submissions_count', 'avg_submission_delay']
for col in submission_cols:
    if col in master_updated.columns:
        before_fill = master_updated[col].isna().sum()
        master_updated[col] = master_updated[col].fillna(0)
        after_fill = master_updated[col].isna().sum()
        print(f"\n{col}: Filled {before_fill} NaN values")

print(f"\nFinal missing values:")
print(master_updated[submission_cols].isnull().sum())

# Save updated master
MERGED_DIR.mkdir(parents=True, exist_ok=True)
master_updated.to_csv(MERGED_FILE, index=False)

print(f"\n✓ Updated master dataframe saved to {MERGED_FILE}")
print(f"Final shape: {master_updated.shape}")
print(f"\nSubmission metrics summary:")
print(master_updated[submission_cols].describe())

MERGING SUBMISSION METRICS INTO MASTER DATAFRAME



Loaded existing merged dataframe: (32548, 18)
Master columns before merge: ['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'date_registration', 'date_unregistration', 'total_clicks', 'num_sites', 'module_presentation_length', 'tma_cma_weighted_score']

Shape after merge: (32548, 21)
New columns: ['total_submissions', 'late_submissions_count', 'avg_submission_delay']

total_submissions: Filled 6709 NaN values

late_submissions_count: Filled 6709 NaN values

avg_submission_delay: Filled 6709 NaN values

Final missing values:
total_submissions         0
late_submissions_count    0
avg_submission_delay      0
dtype: int64

✓ Updated master dataframe saved to ..\..\data\merged\merged.csv
Final shape: (32548, 21)

Submission metrics summary:
       total_submissions  late_submissions_count  avg_submission_delay
count       32548.000000            32548